# DL_DOA_CLONE — Colab reproduction, driven by a pre-generated Kaggle dataset

This notebook reproduces the paper

> D. Lloria, S. Roger, C. Botella-Mascarell, M. Cobos, **"Deep-Learning-Based AoA and AoD
> Estimation in Analog Millimeter Wave MIMO Systems,"** IEEE Trans. Veh. Technol., vol. 75,
> no. 6, pp. 10200–10210, June 2026.

using the `DL_DOA_CLONE` code + pretrained ResNet/U-Net weights, but **every synthetic
observation matrix is read from a dataset you already generated and uploaded to Kaggle** —
nothing is generated on the fly in this notebook.

**What changed vs. the original notebook**

1. The dataset-generation cells are gone. Section 4 below downloads *your* Kaggle
   dataset (the same `dataset/` folder produced by
   `dldoa_dataset_generation.py --mode save_all ...`, i.e. `test_data.npz`,
   `test_meta.npz`, `test_features.npy`, etc.) and loads it from disk.
2. Nothing is hidden inside `z_resnet/main.py` / `z_unet/main.py` anymore. The full
   blob-detection → peak-ranking → Hungarian-matching → PD/RMSE pipeline (paper
   Section III-E and Eq. 12) is written out explicitly, cell by cell, so you can read,
   verify and tweak every step.
3. Every metric the paper reports is computed: PD & RMSE vs. SNR (Figs. 5–6), the
   per-SNR error boxplots (Fig. 7), PD/RMSE vs. number of paths *L* (Fig. 8),
   the hardware-impairment table (Table II), and the asymptotic complexity figures
   (Section IV-E / Tables III–IV).
4. Figs. 8 and Table II need **extra** test splits (different SNR/L/δmax
   combinations) that are not automatically inside a "vanilla" `save_all` export.
   Those sections auto-detect whether you included such splits in your Kaggle
   dataset and clearly say so if they can't find them — they never silently fall
   back to generating data.

**Honesty note on fidelity:** the paper does not publish the exact blob-detector
area thresholds (`minArea`/`maxArea`) used by `cv::SimpleBlobDetector`, only that
shape filters are off and the threshold sweeps the full 0–255 range (Sec. IV-E).
The values used below are reasonable defaults for a 256×256 map with σ=0.07
Gaussians; if your PD numbers look off, tune `get_blob_detector(...)` in Section 6.
Everything else (grid geometry, angle inversion, Hungarian matching, PD/RMSE
definitions) is implemented directly from the paper's equations and cross-checked
against the authors' own dataset-generation code.

**Runtime**: model inference is batched, but blob detection is an inherent
per-sample OpenCV loop (same bottleneck the paper's own repo has). Expect the full
8 000-sample SNR sweep to take a while per model on CPU — a `MAX_SAMPLES_PER_SNR`
knob is provided in Section 7 so you can smoke-test with a small subset first.


## 1. Clone the repo & install dependencies

We still clone `DL_DOA_CLONE` because we need (a) the exact `Resnet`/`UNet` Keras
class definitions the pretrained weights were trained with, and (b) the pretrained
`.h5` weight files themselves. Neither of those is "data generation" — they are the
trained-model artifacts, so this step stays.

In [ ]:
!git clone https://github.com/Mishatmilon059/DL_DOA_CLONE.git
%cd DL_DOA_CLONE
!pip install -q -r requirements.txt


## 2. Extract the U-Net weights (`.7z` parts)

`DL_DOA/models/inf_model_007_256_unet.h5` is split into five `.7z` parts to respect
GitHub's 100 MB limit. The ResNet weights need no extraction.

In [ ]:
!apt-get -qq install -y p7zip-full > /dev/null
%cd DL_DOA/models
!7z x -y inf_model_007_256_unet.7z.001
!ls -la
%cd /content/DL_DOA_CLONE


## 3. Build the ResNet & U-Net architectures and load the pretrained weights

This is already explicit Python (not a black-box script call) in the original
notebook, so it is kept as-is: import the authors' `Resnet`/`UNet` classes and load
the `.h5` weights into them.

In [ ]:
import os, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.append('DL_DOA')
os.chdir('DL_DOA')

import tensorflow as tf
from src.tvt_models import Resnet, UNet

resnet_model = Resnet(input_shape=(64, 64, 2))
resnet_model.load_weights('models/inf_model_007_256_resnet.h5')
print('ResNet weights loaded OK, params:', resnet_model.count_params())

unet_model = UNet(M=256)
unet_model.load_weights('models/inf_model_007_256_unet.h5')
print('UNet weights loaded OK, params:', unet_model.count_params())


## 4. Connect to Kaggle and download *your* dataset

This replaces every dataset-*generation* cell of the original notebook. It expects
your Kaggle dataset to contain the files produced by

```bash
python dldoa_dataset_generation.py --mode save_all --output_dir dataset --n_train 10000 --n_test_per_snr 1000
```

i.e. (at least) `test_data.npz`, `test_meta.npz`, `test_features.npy` (or
`.pkl`) for the main L=3, SNR∈{-10,...,25} sweep used for Figs. 5–7. If you also
generated extra splits for Fig. 8 (L sweep) or Table II (hardware impairment),
just include them anywhere inside the same Kaggle dataset — Section 9/10 will
auto-detect them by inspecting their `*_meta.npz` content.

**Edit `KAGGLE_DATASET` below** to `"<your-kaggle-username>/<your-dataset-slug>"`.

Credentials: this cell first tries Colab **Secrets** (`KAGGLE_USERNAME` /
`KAGGLE_KEY`, set them in the key icon on the left sidebar); if those aren't set,
it asks you to upload your `kaggle.json` API token instead
(Kaggle → Account → *Create New API Token*).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Using Kaggle credentials from Colab Secrets.")
except Exception:
    print("Colab secrets KAGGLE_USERNAME/KAGGLE_KEY not found.")
    print("Please upload your kaggle.json (Kaggle -> Account -> Create New API Token).")
    from google.colab import files
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fn, content in uploaded.items():
        if fn.endswith('.json'):
            with open('/root/.kaggle/kaggle.json', 'wb') as f:
                f.write(content)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle


In [ ]:
# <<< EDIT THIS to your own Kaggle dataset identifier >>>
KAGGLE_DATASET = "your-kaggle-username/your-dataset-slug"

DATA_DIR = "/content/dldoa_dataset"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip

print("\nDownloaded/unzipped contents:")
for root, _, filenames in os.walk(DATA_DIR):
    for fn in filenames:
        print(" ", os.path.relpath(os.path.join(root, fn), DATA_DIR))


## 5. Discover and load the dataset splits actually present

Rather than hard-coding filenames, this scans `DATA_DIR` recursively for any
`<prefix>_data.npz` file and pairs it with the sibling `<prefix>_meta.npz` /
`<prefix>_features.npy|.pkl` / `<prefix>_gt.npz` in the same folder — matching
exactly the naming convention of `dldoa_dataset_generation.py`'s
`save_validation_dataset` / `save_test_dataset`. Each discovered split is then
classified from its meta content: a single fixed `L` with many SNR values is an
"SNR sweep" (Figs. 5–7 data); several `L` values at one or two SNRs is an
"L sweep" (Fig. 8 data).

In [ ]:
import glob
import pickle
import numpy as np

def find_splits(root):
    '''Recursively pair up <prefix>_data.npz with its sibling files.'''
    splits = {}
    for data_path in glob.glob(os.path.join(root, '**', '*_data.npz'), recursive=True):
        d = os.path.dirname(data_path)
        base = os.path.basename(data_path)
        if not base.endswith('_data.npz'):
            continue
        prefix = base[: -len('_data.npz')]
        entry = {'data': data_path}
        meta_path = os.path.join(d, f'{prefix}_meta.npz')
        if os.path.exists(meta_path):
            entry['meta'] = meta_path
        feat_npy = os.path.join(d, f'{prefix}_features.npy')
        feat_pkl = os.path.join(d, f'{prefix}_features.pkl')
        if os.path.exists(feat_npy):
            entry['features'] = feat_npy
        elif os.path.exists(feat_pkl):
            entry['features'] = feat_pkl
        gt_path = os.path.join(d, f'{prefix}_gt.npz')
        if os.path.exists(gt_path):
            entry['gt'] = gt_path
        rel_dir = os.path.relpath(d, root)
        key = prefix if rel_dir == '.' else f'{rel_dir}/{prefix}'
        splits[key] = entry
    return splits


def load_split(entry):
    '''Load (data, meta, features) for one discovered split.'''
    data = np.load(entry['data'])['data']
    meta = np.load(entry['meta'])['data'] if 'meta' in entry else None
    features = None
    if 'features' in entry:
        fpath = entry['features']
        if fpath.endswith('.pkl'):
            with open(fpath, 'rb') as f:
                features = pickle.load(f)
        else:
            features = np.load(fpath, allow_pickle=True)
    return data, meta, features


def classify_split(meta):
    '''Heuristic: figure out which paper figure this split corresponds to.'''
    if meta is None:
        return 'unknown (no *_meta.npz found)'
    L_vals = np.unique(meta[:, 0])
    snr_vals = np.unique(meta[:, 1])
    if len(L_vals) == 1 and len(snr_vals) >= 5:
        return 'snr_sweep'      # Figs. 5-7 style: fixed L, many SNR points
    if len(L_vals) >= 3 and len(snr_vals) <= 3:
        return 'l_sweep'        # Fig. 8 style: several L, one/two SNR points
    return 'unknown'


splits = find_splits(DATA_DIR)
print(f"Found {len(splits)} candidate split(s) under {DATA_DIR}:\n")
split_info = {}
for name, entry in splits.items():
    _, meta, _ = load_split(entry)
    kind = classify_split(meta)
    split_info[name] = kind
    extra = ''
    if meta is not None:
        extra = f" | L={sorted(set(meta[:,0].tolist()))} SNR={sorted(set(meta[:,1].tolist()))} n={len(meta)}"
    print(f"  - {name:30s} [{kind}]{extra}")


In [ ]:
# Pick the main SNR-sweep split (paper Figs. 5-7: L=3, nt=nr=Q=P=16, SNR -10..25 step 5)
MAIN_SPLIT = next((n for n, k in split_info.items() if k == 'snr_sweep'), None)
if MAIN_SPLIT is None:
    raise RuntimeError(
        "Could not find an SNR-sweep test split (fixed L, >=5 SNR values) in your "
        "Kaggle dataset. Make sure it contains 'test_data.npz' + 'test_meta.npz' + "
        "'test_features.npy' (or .pkl) as produced by "
        "'dldoa_dataset_generation.py --mode save_all'."
    )
print("Using main SNR-sweep split:", MAIN_SPLIT)

test_data, test_meta, test_features = load_split(splits[MAIN_SPLIT])
print("test_data   shape:", test_data.shape)
print("test_meta   shape:", test_meta.shape, "columns = [L, SNR, P, nt]")
print("test_features: list/array of", len(test_features), "entries, each shape (2, L) = [psi_l; phi_l]")


## 6. The evaluation pipeline (paper Section III-E & Eq. 12), written explicitly

This block implements, from scratch and fully inspectable:

1. **Preprocessing** — normalize the network's predicted map to 8-bit grayscale.
2. **Blob detection** — `cv2.SimpleBlobDetector` with shape filters (circularity /
   convexity / inertia) switched off and the threshold swept 0–255, exactly as
   Sec. IV-E describes.
3. **Peak localization** — each blob's (subpixel) centroid pixel is converted back
   to a spatial frequency and then to an angle, inverting Eqs. (9)–(10) and the
   grid/wrapping convention of Eq. (11). *(This inversion was numerically
   validated against the authors' own `generate_gt` grid-construction code before
   being used here — see the notebook's build notes.)*
4. **Peak ranking** — if more blobs than paths are found, keep the `L` most
   intense ones (as amplitude-ranking, Sec. III-E step 4).
5. **Matching + Eq. (12)** — optimal (Hungarian) assignment between estimated and
   true `(psi, phi)` pairs; each individual AoA/AoD is "detected" if its angular
   error is `<= 1 deg`; **Pd** is the fraction of the `2*L` true angles per
   condition that are detected, and **RMSE** is computed only over that detected
   set, exactly as Eq. (12) and the surrounding text define it.

Blob-detector area thresholds are not published in the paper; the defaults below
are reasonable for a 256x256, sigma=0.07 heat-map — tune `min_area`/`max_area` if
detections look too fragmented or too merged.

In [ ]:
import cv2
from scipy.optimize import linear_sum_assignment

GRID_SIGMA = 0.07        # paper: sigma1 = sigma2 = 0.07 (Table I, Eq. 11)
MARGIN_FACTOR = 3.0      # matches the authors' generate_gt() grid margin
GRID_SIZE = 256          # paper: M = N = 256


def _pixel_to_omega(idx, num_points, sigma=GRID_SIGMA, margin_factor=MARGIN_FACTOR):
    '''Inverse of the linspace(-margin, 2*pi+margin, num_points, endpoint=False)
    grid used by generate_gt() to build the ground-truth heat-map (Eq. 11).'''
    margin = margin_factor * sigma
    step = (2 * np.pi + 2 * margin) / num_points
    return -margin + idx * step


def _omega_to_phi(w):
    '''Invert omega_phi = pi*cos(phi) after Eq.(11)'s wrapTo2Pi (Eqs. 9-10).'''
    orig = np.where(w > np.pi, w - 2 * np.pi, w)
    return np.arccos(np.clip(orig / np.pi, -1.0, 1.0))


def _omega_to_psi(w):
    '''Invert omega_psi = -pi*cos(psi) after wrapTo2Pi (Eqs. 9-10).'''
    orig = np.where(w > np.pi, w - 2 * np.pi, w)
    return np.arccos(np.clip(-orig / np.pi, -1.0, 1.0))


def get_blob_detector(min_area=4, max_area=6000, min_threshold=0, max_threshold=255,
                       threshold_step=10):
    '''Sec. III-E preprocessing/blob-detection stages: bright blobs only, no
    shape filters, full 0-255 threshold sweep.'''
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True
    p.blobColor = 255
    p.filterByArea = True
    p.minArea = min_area
    p.maxArea = max_area
    p.filterByCircularity = False
    p.filterByConvexity = False
    p.filterByInertia = False
    p.minThreshold = min_threshold
    p.maxThreshold = max_threshold
    p.thresholdStep = threshold_step
    p.minDistBetweenBlobs = 1
    return cv2.SimpleBlobDetector_create(p)


def get_blob_peaks(pred_img, detector):
    '''Normalize to 8-bit grayscale and run the blob detector (Sec III-E, 1-2).'''
    img = np.asarray(pred_img, dtype=np.float32).squeeze()
    mn, mx = float(img.min()), float(img.max())
    if mx - mn < 1e-12:
        return np.zeros((0, 2)), np.zeros((0,))
    norm = ((img - mn) / (mx - mn) * 255.0).astype(np.uint8)
    kps = detector.detect(norm)
    if len(kps) == 0:
        return np.zeros((0, 2)), np.zeros((0,))
    peaks = np.array([[kp.pt[1], kp.pt[0]] for kp in kps])  # (row, col), subpixel
    amps = np.array([img[int(round(kp.pt[1])), int(round(kp.pt[0]))] for kp in kps])
    return peaks, amps


def peaks_to_angles(peaks, sigma=GRID_SIGMA, grid_size=GRID_SIZE):
    '''Peak localization (Sec III-E, step 3): pixel -> (psi, phi) in radians.'''
    if len(peaks) == 0:
        return np.zeros((0, 2))
    r, c = peaks[:, 0], peaks[:, 1]
    psi = _omega_to_psi(_pixel_to_omega(r, grid_size, sigma))
    phi = _omega_to_phi(_pixel_to_omega(c, grid_size, sigma))
    return np.stack([psi, phi], axis=1)


def match_and_score(est_angles, true_psi, true_phi, thresh_deg=1.0):
    '''Hungarian matching + the RMSE/Pd criterion of Eq. (12).
    Returns (errors_deg, is_good, n_true_angles) where n_true_angles = 2*L
    (L AoAs + L AoDs) is the fixed Pd denominator for this sample.'''
    true_pts = np.stack([true_psi, true_phi], axis=1)
    L = len(true_pts)
    if len(est_angles) == 0:
        return np.array([]), np.array([], dtype=bool), 2 * L
    dist = np.linalg.norm(est_angles[:, None, :] - true_pts[None, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(dist)
    psi_err = np.degrees(np.abs(est_angles[row_ind, 0] - true_pts[col_ind, 0]))
    phi_err = np.degrees(np.abs(est_angles[row_ind, 1] - true_pts[col_ind, 1]))
    errs = np.concatenate([psi_err, phi_err])
    return errs, errs <= thresh_deg, 2 * L


In [ ]:
def evaluate_over_dataset(model, data_arr, meta_arr, features_list, group_by='SNR',
                            fixed_filters=None, thresh_deg=1.0, predict_batch_size=64,
                            max_samples_per_group=None, verbose_every=1000):
    '''
    Runs the full pipeline over a loaded split and aggregates Pd/RMSE per group.

    group_by         : 'L' or 'SNR' -- which meta column to report results by
                        (meta columns are [L, SNR, P, nt]).
    fixed_filters     : optional {col_index: value} to restrict to a subset,
                        e.g. {1: 0} to keep only SNR == 0 dB when group_by='L'.
    max_samples_per_group : cap the number of samples used per group value, for
                        quick smoke tests (None = use everything found).

    Returns a dict: group_value -> {'Pd':..., 'RMSE':..., 'n':..., 'raw_errors':...}
    exactly matching Eq. (12) and the Pd definition of Sec. IV.
    '''
    col = {'L': 0, 'SNR': 1, 'P': 2, 'nt': 3}[group_by]

    # -- select indices honoring fixed_filters and the optional per-group cap --
    by_group = {}
    for i in range(len(meta_arr)):
        if fixed_filters and any(meta_arr[i][c] != v for c, v in fixed_filters.items()):
            continue
        key = float(meta_arr[i][col])
        by_group.setdefault(key, []).append(i)
    idxs = []
    for key, lst in by_group.items():
        idxs.extend(lst if max_samples_per_group is None else lst[:max_samples_per_group])
    idxs = sorted(idxs)

    # -- batched forward pass (the expensive NN part is NOT done per-sample) --
    preds = model.predict(data_arr[idxs], batch_size=predict_batch_size, verbose=0)

    detector = get_blob_detector()
    good_counts, total_counts, good_errors = {}, {}, {}
    for n, i in enumerate(idxs):
        pred = preds[n, ..., 0]
        feat = np.asarray(features_list[i])       # shape (2, L): [psi_l; phi_l]
        true_psi, true_phi = feat[0], feat[1]
        L = len(true_psi)

        peaks, amps = get_blob_peaks(pred, detector)
        if len(peaks) > L:
            top = np.argsort(-amps)[:L]
            peaks = peaks[top]
        est = peaks_to_angles(peaks)
        errs, good, denom = match_and_score(est, true_psi, true_phi, thresh_deg=thresh_deg)

        key = float(meta_arr[i][col])
        good_counts[key] = good_counts.get(key, 0) + int(good.sum())
        total_counts[key] = total_counts.get(key, 0) + denom
        good_errors.setdefault(key, []).append(errs[good])

        if (n + 1) % verbose_every == 0:
            print(f"    ... {n + 1}/{len(idxs)} samples processed")

    results = {}
    for key in sorted(good_counts):
        pd_ = good_counts[key] / total_counts[key] if total_counts[key] else float('nan')
        e = np.concatenate(good_errors[key]) if good_errors[key] else np.array([])
        rmse = float(np.sqrt(np.mean(e ** 2))) if len(e) else float('nan')
        results[key] = {'Pd': pd_, 'RMSE': rmse, 'n': total_counts[key], 'raw_errors': e}
    return results


## 7. Run the evaluation: Figs. 5–6 data (Pd & RMSE vs. SNR, L=3)

Set `MAX_SAMPLES_PER_SNR` to a small number (e.g. `50`) first to smoke-test the
whole pipeline quickly; set it to `None` for the paper's full 1000-samples/SNR
run (this is the slow, blob-detection-bound part the original notebook warned
about).

In [ ]:
MAX_SAMPLES_PER_SNR = 50   # <<< set to None for the full 1000-samples/SNR paper run

print("=== ResNet: Pd & RMSE vs SNR (L=3, nt=nr=Q=P=16) ===")
resnet_results = evaluate_over_dataset(
    resnet_model, test_data, test_meta, test_features,
    group_by='SNR', max_samples_per_group=MAX_SAMPLES_PER_SNR,
)
for snr, r in resnet_results.items():
    print(f"  SNR={snr:>5.0f} dB -> Pd={r['Pd']*100:6.2f}%  RMSE={r['RMSE']:.4f} deg  (n={r['n']})")

print("\n=== UNet: Pd & RMSE vs SNR (L=3, nt=nr=Q=P=16) ===")
unet_results = evaluate_over_dataset(
    unet_model, test_data, test_meta, test_features,
    group_by='SNR', max_samples_per_group=MAX_SAMPLES_PER_SNR,
)
for snr, r in unet_results.items():
    print(f"  SNR={snr:>5.0f} dB -> Pd={r['Pd']*100:6.2f}%  RMSE={r['RMSE']:.4f} deg  (n={r['n']})")


## 8. Figs. 5–6 style plots: Pd and RMSE vs. SNR

*Scope note:* the paper's Figs. 5–6 also plot the TSDCE, DFT-CEA and CRLB
baselines. Those signal-processing baselines are not part of this notebook (they
were not in the original notebook either) — only the two deep-learning curves are
reproduced here.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

snrs_r = sorted(resnet_results)
snrs_u = sorted(unet_results)

axes[0].plot(snrs_r, [resnet_results[s]['Pd'] * 100 for s in snrs_r], 'o-', label='ResNet')
axes[0].plot(snrs_u, [unet_results[s]['Pd'] * 100 for s in snrs_u], 's-', label='UNet')
axes[0].set_xlabel('SNR (dB)'); axes[0].set_ylabel('Pd (%)')
axes[0].set_title('Probability of detection vs SNR (L=3)')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(snrs_r, [resnet_results[s]['RMSE'] for s in snrs_r], 'o-', label='ResNet')
axes[1].plot(snrs_u, [unet_results[s]['RMSE'] for s in snrs_u], 's-', label='UNet')
axes[1].set_xlabel('SNR (dB)'); axes[1].set_ylabel('RMSE (deg)')
axes[1].set_title('RMSE vs SNR (L=3, detected angles only)')
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Fig. 7 style plot: boxplots of angle estimation errors

Uses the raw per-angle errors of the *detected* angles (`raw_errors`, already
collected by `evaluate_over_dataset`) at each SNR — same data source as the RMSE
numbers above, just visualized as a distribution instead of a single number.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, results, title in [(axes[0], resnet_results, 'ResNet'), (axes[1], unet_results, 'UNet')]:
    snrs = sorted(results)
    data = [results[s]['raw_errors'] for s in snrs]
    ax.boxplot(data, labels=[f"{s:.0f}" for s in snrs], showfliers=False)
    ax.set_xlabel('SNR (dB)'); ax.set_title(f'{title}: angle error distribution (L=3)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Angle error (deg)')
plt.tight_layout()
plt.show()


## 10. Fig. 8 style data: Pd & RMSE vs. number of paths *L*

This needs a **different** test split (several `L` values at SNR in {0, 20} dB,
nt=nr=Q=P=16) — i.e. one generated with
`generate_test_conditions_fig8()` / a matching `save_test_dataset(...)` call in
the original pipeline. The cell below only runs if such a split was detected in
Section 5; otherwise it explains exactly what to add to your Kaggle dataset,
rather than silently generating anything.

In [ ]:
FIG8_SPLIT = next((n for n, k in split_info.items() if k == 'l_sweep'), None)

if FIG8_SPLIT is None:
    print("No L-sweep split found in your Kaggle dataset (expected: several L values "
          "at SNR in {0, 20} dB, nt=nr=Q=P=16 -- i.e. data generated with "
          "generate_test_conditions_fig8()).")
    print("Skipping Fig. 8 reproduction. To enable it, add a split such as "
          "'fig8_data.npz' + 'fig8_meta.npz' + 'fig8_features.npy' built from "
          "that condition list to your Kaggle dataset and re-run Section 5.")
else:
    print("Using L-sweep split:", FIG8_SPLIT)
    fig8_data, fig8_meta, fig8_features = load_split(splits[FIG8_SPLIT])
    snr_values_found = sorted(set(fig8_meta[:, 1].tolist()))
    print("SNR values found in this split:", snr_values_found)

    fig8_resnet, fig8_unet = {}, {}
    for snr in snr_values_found:
        fig8_resnet[snr] = evaluate_over_dataset(
            resnet_model, fig8_data, fig8_meta, fig8_features,
            group_by='L', fixed_filters={1: snr}, max_samples_per_group=MAX_SAMPLES_PER_SNR,
        )
        fig8_unet[snr] = evaluate_over_dataset(
            unet_model, fig8_data, fig8_meta, fig8_features,
            group_by='L', fixed_filters={1: snr}, max_samples_per_group=MAX_SAMPLES_PER_SNR,
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for snr in snr_values_found:
        Ls = sorted(fig8_resnet[snr])
        axes[0].plot(Ls, [fig8_resnet[snr][l]['Pd'] * 100 for l in Ls], 'o-', label=f'ResNet {snr:.0f} dB')
        axes[0].plot(Ls, [fig8_unet[snr][l]['Pd'] * 100 for l in Ls], 's--', label=f'UNet {snr:.0f} dB')
        axes[1].plot(Ls, [fig8_resnet[snr][l]['RMSE'] for l in Ls], 'o-', label=f'ResNet {snr:.0f} dB')
        axes[1].plot(Ls, [fig8_unet[snr][l]['RMSE'] for l in Ls], 's--', label=f'UNet {snr:.0f} dB')
    axes[0].set_xlabel('Number of paths L'); axes[0].set_ylabel('Pd (%)')
    axes[0].set_title('Pd vs L'); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
    axes[1].set_xlabel('Number of paths L'); axes[1].set_ylabel('RMSE (deg)')
    axes[1].set_title('RMSE vs L'); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()


## 11. Table II style data: robustness to hardware (phase-shifter) impairments

Table II compares Pd/RMSE at SNR ∈ {0, 20} dB across four phase-error levels
`delta_max` ∈ {0°, 1°, 2°, 5°} (Sec. IV-D). Because the saved `*_meta.npz` files
do **not** record `delta_max` (it only perturbs how the data was generated, it is
not part of the condition tuple), these four datasets can only be told apart by
where/how you name them in your Kaggle dataset.

Edit `HARDWARE_SPLIT_PREFIXES` below to map each δmax level to the split name
Section 5 printed for it (e.g. if you uploaded `hw_err0_data.npz`,
`hw_err1_data.npz`, ... the prefixes are `hw_err0`, `hw_err1`, ...). Any level
left as `None` (or not found) is skipped with a clear message — nothing is
generated to fill the gap.

In [ ]:
HARDWARE_SPLIT_PREFIXES = {
    0: None,   # e.g. "hw_err0"
    1: None,   # e.g. "hw_err1"
    2: None,   # e.g. "hw_err2"
    5: None,   # e.g. "hw_err5"
}

hardware_rows = []
for deg, prefix in HARDWARE_SPLIT_PREFIXES.items():
    if prefix is None or prefix not in splits:
        print(f"delta_max={deg} deg: no split configured/found -- skipping.")
        continue
    hw_data, hw_meta, hw_features = load_split(splits[prefix])
    for snr in (0, 20):
        r_res = evaluate_over_dataset(resnet_model, hw_data, hw_meta, hw_features,
                                       group_by='SNR', fixed_filters={1: snr},
                                       max_samples_per_group=MAX_SAMPLES_PER_SNR)
        r_un = evaluate_over_dataset(unet_model, hw_data, hw_meta, hw_features,
                                      group_by='SNR', fixed_filters={1: snr},
                                      max_samples_per_group=MAX_SAMPLES_PER_SNR)
        if snr in r_res:
            hardware_rows.append({
                'delta_max (deg)': deg, 'SNR (dB)': snr,
                'ResNet Pd (%)': r_res[snr]['Pd'] * 100, 'ResNet RMSE': r_res[snr]['RMSE'],
                'UNet Pd (%)': r_un[snr]['Pd'] * 100, 'UNet RMSE': r_un[snr]['RMSE'],
            })

if hardware_rows:
    import pandas as pd
    display(pd.DataFrame(hardware_rows))
else:
    print("\nNo hardware-impairment splits configured -- Table II is skipped. "
          "Fill in HARDWARE_SPLIT_PREFIXES above once those splits are part of "
          "your Kaggle dataset.")


## 12. Complexity analysis (Section IV-E / Tables III–IV)

This section is purely analytical (no dataset needed) and reproduces the
asymptotic complexity formulas exactly as derived in Sec. IV-E:

* ResNet / U-Net (pre-peak-detection stage): **O(P'Q')**, i.e. **O(PQ)** since the
  upsampling factor `beta` is a constant.
* Peak detection (blob scan + amplitude ranking): **O(P'Q')** dominates
  **O(L log L)** whenever `L < {P, Q}` (always true here), so it is also **O(PQ)**.
* TSDCE baseline: **O((L-1) * P * Q * min(P, Q))**.
* DFT-CEA baseline: **O(N_DFT^2 * log2(N_DFT))**, with `N_DFT = 1024` in the paper.

In [ ]:
import math

def complexity_nn(P, Q):
    '''ResNet / U-Net + peak-detection asymptotic complexity: O(P*Q).'''
    return P * Q

def complexity_tsdce(P, Q, L):
    '''O((L-1) * P * Q * min(P, Q)).'''
    return max(L - 1, 0) * P * Q * min(P, Q)

def complexity_dft_cea(N_DFT=1024):
    '''O(N_DFT^2 * log2(N_DFT)).'''
    return (N_DFT ** 2) * math.log2(N_DFT)

print(f"{'P=Q':>6} {'L':>3} {'NN (P*Q)':>14} {'TSDCE':>16} {'DFT-CEA (N=1024)':>18}")
for P in (16, 32, 64):
    for L in (1, 3, 6, 9):
        nn = complexity_nn(P, P)
        ts = complexity_tsdce(P, P, L)
        dc = complexity_dft_cea()
        print(f"{P:>6} {L:>3} {nn:>14,} {ts:>16,} {dc:>18,.0f}")


## 13. ResNet vs. UNet summary table & comparison plot

In [ ]:
import pandas as pd

snrs = sorted(resnet_results)
summary_rows = []
for snr in snrs:
    summary_rows.append({
        'SNR (dB)': snr,
        'ResNet Pd (%)': f"{resnet_results[snr]['Pd']*100:.2f}",
        'UNet Pd (%)': f"{unet_results[snr]['Pd']*100:.2f}",
        'ResNet RMSE (deg)': f"{resnet_results[snr]['RMSE']:.4f}",
        'UNet RMSE (deg)': f"{unet_results[snr]['RMSE']:.4f}",
    })
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(snrs, [resnet_results[s]['Pd']*100 for s in snrs], 'o-', label='ResNet')
ax[0].plot(snrs, [unet_results[s]['Pd']*100 for s in snrs], 's-', label='UNet')
ax[0].set_title('Pd comparison'); ax[0].set_xlabel('SNR (dB)'); ax[0].set_ylabel('Pd (%)')
ax[0].grid(alpha=0.3); ax[0].legend()

ax[1].plot(snrs, [resnet_results[s]['RMSE'] for s in snrs], 'o-', label='ResNet')
ax[1].plot(snrs, [unet_results[s]['RMSE'] for s in snrs], 's-', label='UNet')
ax[1].set_title('RMSE comparison'); ax[1].set_xlabel('SNR (dB)'); ax[1].set_ylabel('RMSE (deg)')
ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()


## Notes & limitations

- **Blob-detector parameters** (`min_area`, `max_area` in `get_blob_detector`,
  Section 6) are reasonable defaults, not values published in the paper. If Pd
  looks too low (blobs merging / fragmenting) or too high (spurious tiny blobs
  counted as peaks), tune them and re-run Section 7.
- **Signal-processing baselines** (TSDCE, DFT-CEA) and the **CRLB** curve shown
  alongside Figs. 5–6 in the paper are *not* implemented here — this notebook
  reproduces only the two deep-learning models' own numbers, exactly like the
  original notebook did.
- **Figs. 8 and Table II** need test splits with different SNR/L/δmax
  combinations than the default `save_all` export. Sections 10–11 auto-detect
  what's available in your Kaggle dataset and clearly report what is missing
  instead of generating substitute data.
- All angle-recovery math in Section 6 (grid geometry, wrap handling, the
  pixel→angle inversion) was numerically validated against the authors' own
  `generate_gt()` grid-construction code before being used for evaluation.
